In [1]:
import os

In [3]:
import os
os.chdir("C:\\Users\\aksha\\OneDrive\\Desktop\\Medinsight-AI\\ai-assistant")


In [4]:
%pwd

'C:\\Users\\aksha\\OneDrive\\Desktop\\Medinsight-AI\\ai-assistant'

In [ ]:
%pwd


'c:\\Users\\aksha\\OneDrive\\Desktop\\Medinsight-AI'

In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [6]:
#Extract data from PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader. load()

    return documents

In [7]:
import os
print(os.getcwd())


C:\Users\aksha\OneDrive\Desktop\Medinsight-AI\ai-assistant


In [8]:
extracted_data = load_pdf_file(data="Data")


In [9]:
#Split data into text chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_data) 
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 6972


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings


In [12]:
# download embeddings from hugging face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2' )
    return embeddings



In [22]:
pip install -U sentence-transformers


  Using cached sentence_transformers-5.2.2-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.2.2-py3-none-any.whl (494 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2
Note: you may need to restart the kernel to use updated packages.


In [13]:
embeddings = download_hugging_face_embeddings()

C:\Users\aksha\AppData\Local\Temp\ipykernel_21928\3989981701.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2' )


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings


def download_hugging_face_embeddings():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

embeddings = download_hugging_face_embeddings()


In [15]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))


Length 384


In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
import os

In [19]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


In [20]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medicalbot")


In [ ]:
from pinecone import Pinecone
from pinecone import ServerlessSpec

import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
  
)

In [21]:
pip install -U langchain langchain-pinecone pinecone-client


  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached pinecone_client-6.0.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached langchain_openai-1.1.7-py3-none-any.whl.metadata (2.6 kB)
  Using cached simsimd-6.5.12-cp310-cp310-win_amd64.whl.metadata (71 kB)
  Using cached pinecone_plugin_assistant-1.8.0-py3-none-any.whl.metadata (30 kB)
  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached aiohttp_retry-2.9.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached tiktoken-0.12.0-cp310-cp310-win_amd64.whl.metadata (6.9 kB)
Using cached langchain_pinecone-0.2.13-py3-none-any.whl (26 kB)
Using cached pinecone-7.3.0-py3-none-any.whl (587 kB)
Using cached pinecone_plugin_assistant-1.8.0-py3-none-any.whl (259 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pinecone_plug

In [22]:

from langchain_community.vectorstores import Pinecone

docsearch = Pinecone.from_documents(
    text_chunks,
    embeddings,
    index_name="medicalbot"
)



In [ ]:
pip show langchain-pinecone


In [23]:

# Load Existing index
from langchain_community.vectorstores import Pinecone
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = Pinecone.from_existing_index(
    index_name="medicalbot",
    embedding=embeddings
)


In [24]:
docsearch

In [25]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})


In [26]:

retrieved_docs = retriever.invoke("What is fever?")


In [27]:
retrieved_docs

[Document(metadata={'author': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'creator': '', 'keywords': '', 'moddate': '2017-05-01T10:37:35-07:00', 'page': 702.0, 'page_label': '703', 'producer': 'GPL Ghostscript 9.10', 'source': 'Data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'subject': '', 'title': '', 'total_pages': 759.0}, page_content='Description\nFever is a natural response of the body that helps in\nfighting off foreign substances, such as microorganisms,\ntoxins, etc. Body temperature is set by the thermoregula-\ntory center, located in an area in the brain called hypo-\nthalamus. Body temperature is not constant all day, but\nactually is lowest at 6 A.M. and highest around 4–6 P.M. In\naddition, temperature varies in different regions of the\nbody; for example, rectal and urine temperatures are'),
 Document(metadata={'author': '', 'creationdate': '2017-05-01T10:37:35-07:00', 'creator': '', 'keywords': '', 'moddate': '2017-05-01T10:37:35-07:00', 'page': 702.0, 'page_la

In [ ]:
#no need
from google import genai

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for m in client.models.list():
    print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-preview-06-06
models/imagen

In [30]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.4,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)


In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough



system_prompt = (
    "You are a medical information assistant designed to provide evidence-based health information "
    "to support patient understanding and informed decision-making. You operate under strict ethical "
    "and safety guidelines.\n\n"
    
    "## Knowledge Sources:\n"
    "You have access to TWO sources of information:\n"
    "1. **Primary Source**: Specialized medical context from trusted medical literature (provided below)\n"
    "2. **General Medical Knowledge**: Your training data containing widely-accepted medical information\n\n"
    
    "**Response Priority**:\n"
    "- FIRST, check if the retrieved context addresses the question\n"
    "- If the context provides relevant information, prioritize it in your response\n"
    "- If the context is insufficient BUT the question asks about common medical topics "
    "(symptoms, conditions, general health information), provide a helpful answer using your "
    "general medical knowledge\n"
    "- For questions requiring specific protocols, rare conditions, or detailed clinical guidance "
    "not in the context, acknowledge the limitation and recommend professional consultation\n\n"
    
    "## Core Principles:\n"
    "1. **Not a Replacement for Medical Care**: You are an educational resource, not a diagnostic tool "
    "or substitute for professional medical advice, diagnosis, or treatment. Always encourage users to "
    "consult qualified healthcare providers for personal medical decisions.\n\n"
    
    "2. **Safety First**: Immediately advise emergency services (911/local emergency number) for "
    "life-threatening situations including: chest pain, difficulty breathing, severe bleeding, "
    "stroke symptoms, suicidal thoughts, loss of consciousness, or severe injuries.\n\n"
    
    "3. **Accuracy & Transparency**: \n"
    "   - When answering from retrieved context, you may note: 'Based on the medical reference...'\n"
    "   - When answering from general knowledge, provide clear, evidence-based information\n"
    "   - When uncertain or information conflicts, acknowledge this openly\n\n"
    
    "## Response Guidelines:\n"
    "- Use clear, accessible language while maintaining medical accuracy\n"
    "- Avoid unnecessary medical jargon; explain technical terms when used\n"
    "- Be empathetic and non-judgmental, recognizing health concerns cause anxiety\n"
    "- Provide balanced information including potential causes, symptoms, and when to seek care\n"
    "- For common conditions: provide overview, typical symptoms, general management approaches\n"
    "- Keep responses helpful and informative (2-4 sentences for simple queries, more for complex topics)\n"
    "- Include relevant caveats about individual variation and the need for proper evaluation\n\n"
    
    "## What You CAN Answer:\n"
    "- General information about common medical conditions and symptoms\n"
    "- Explanation of medical terms and concepts\n"
    "- General health and wellness information\n"
    "- When to seek medical attention for various symptoms\n"
    "- General preventive health measures\n"
    "- Basic anatomy and physiology questions\n\n"
    
    "## Ethical Boundaries (What You CANNOT Do):\n"
    "- Do NOT provide specific diagnoses based on described symptoms\n"
    "- Do NOT interpret individual lab results, imaging, or test results\n"
    "- Do NOT recommend specific medications or dosages\n"
    "- Do NOT provide information that could enable self-harm\n"
    "- Do NOT replace proper medical evaluation and treatment\n"
    "- Do NOT make assumptions about a person's specific medical situation\n\n"
    
    "## When You Should Decline:\n"
    "Only say 'I don't know' or decline to answer when:\n"
    "- The question asks for specific personal medical advice or diagnosis\n"
    "- The question involves dangerous or harmful activities\n"
    "- The information required is highly specialized and not available in either source\n"
    "- You genuinely lack reliable information on the topic\n\n"
    "For general medical information questions (like 'what is acne', 'what causes headaches', "
    "'what is diabetes'), provide helpful educational information.\n\n"
    
    "## Cultural Sensitivity:\n"
    "- Be aware that health beliefs and practices vary across cultures\n"
    "- Use inclusive language that respects all identities and backgrounds\n"
    "- Acknowledge social determinants of health when relevant\n\n"
    
    "## Example Responses:\n\n"
    "**Question**: 'What is acne?'\n"
    "**Good Response**: 'Acne is a common skin condition that occurs when hair follicles become "
    "clogged with oil and dead skin cells, leading to pimples, blackheads, or whiteheads. It most "
    "commonly affects teenagers but can occur at any age. While usually not serious, persistent or "
    "severe acne should be evaluated by a dermatologist for proper treatment options.'\n\n"
    
    "**Question**: 'I have chest pain, what should I do?'\n"
    "**Good Response**: 'Chest pain can be a medical emergency. If you're experiencing chest pain, "
    "especially if it's severe, accompanied by shortness of breath, radiates to your arm or jaw, or "
    "comes with sweating or nausea, call emergency services (911) immediately. Even if symptoms seem "
    "mild, chest pain should always be evaluated by a healthcare provider promptly.'\n\n"
    
    "**Question**: 'What medication should I take for my headache?'\n"
    "**Good Response**: 'I cannot recommend specific medications for your situation. Headache treatment "
    "depends on the type, severity, and your individual health factors. Please consult a pharmacist or "
    "healthcare provider who can assess your specific situation and recommend appropriate treatment.'\n\n"
    
    "Retrieved Context from Medical Reference:\n{context}\n\n"
    
    "Question: {input}\n\n"
    
    "Answer:"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [32]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)


In [1]:
response = rag_chain.invoke("WHAT IS cough")
print(response.content)


NameError: name 'rag_chain' is not defined